In [1]:
import open3d as o3d
import numpy as np
import copy
import os
import sys

# # monkey patches visualization and provides helpers to load geometries
# sys.path.append('..')
# import open3d_tutorial as o3dtut
# # change to True if you want to interact with the visualization windows
# o3dtut.interactive = not "CI" in os.environ

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# ICP registration
This tutorial demonstrates the ICP (Iterative Closest Point) registration algorithm. It has been a mainstay of geometric registration in both research and industry for many years. The input are two point clouds and an initial transformation that roughly aligns the source point cloud to the target point cloud. The output is a refined transformation that tightly aligns the two point clouds. A helper function `draw_registration_result` visualizes the alignment during the registration process. In this tutorial, we show two ICP variants, the point-to-point ICP and the point-to-plane ICP [\[Rusinkiewicz2001\]](../reference.html#rusinkiewicz2001).

## Helper visualization function
The function below visualizes a target point cloud and a source point cloud transformed with an alignment transformation. The target point cloud and the source point cloud are painted with cyan and yellow colors respectively. The more and tighter the two point clouds overlap with each other, the better the alignment result.

In [2]:
def draw_registration_result(source, target, transformation):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])
    source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp],
                                      zoom=0.4459,
                                      front=[0.9288, -0.2951, -0.2242],
                                      lookat=[1.6784, 2.0612, 1.4451],
                                      up=[-0.3402, -0.9189, -0.1996])

<div class="alert alert-info">
    
**Note:** 

Since the functions `transform` and `paint_uniform_color` change the point cloud, we call `copy.deepcopy` to make copies and protect the original point clouds.

</div>

## Input
The code below reads a source point cloud and a target point cloud from two files. A rough transformation is given.

<div class="alert alert-info">
    
**Note:** 

The initial alignment is usually obtained by a global registration algorithm. See [Global registration](../pipelines/global_registration.rst) for examples.

</div>

In [7]:
# source = o3d.io.read_point_cloud("../../data/ICP/cloud_bin_0.pcd")
# target = o3d.io.read_point_cloud("../../data/ICP/cloud_bin_1.pcd")
source = o3d.io.read_point_cloud("D:\REGISTRATION DATA\External\JB-483 70ft EXT\\top_center_aligned_with_radius_extended_ground/057.txt", format='xyz')
target = o3d.io.read_point_cloud("D:\REGISTRATION DATA\External\JB-483 70ft EXT\\top_center_aligned_with_radius_extended_ground/058.txt", format='xyz')
threshold = 0.02
# trans_init = np.asarray([[0.862, 0.011, -0.507, 0.5],
#                          [-0.139, 0.967, -0.215, 0.7],
#                          [0.487, 0.255, 0.835, -1.4], [0.0, 0.0, 0.0, 1.0]])
trans_init = np.identity(4)
# draw_registration_result(source, target, trans_init)

The function `evaluate_registration` calculates two main metrics:

- `fitness`, which measures the overlapping area (# of inlier correspondences / # of points in target). The higher the better.
- `inlier_rmse`, which measures the RMSE of all inlier correspondences. The lower the better.

In [4]:
print("Initial alignment")
evaluation = o3d.pipelines.registration.evaluate_registration(
    source, target, threshold, trans_init)
print(evaluation)

Initial alignment
RegistrationResult with fitness=7.905310e-03, inlier_rmse=1.527898e-02, and correspondence_set size of 1999
Access transformation to get result.


## Point-to-point ICP
In general, the ICP algorithm iterates over two steps:

1. Find correspondence set $\mathcal{K}=\{(\mathbf{p}, \mathbf{q})\}$ from target point cloud $\mathbf{P}$, and source point cloud $\mathbf{Q}$ transformed with current transformation matrix $\mathbf{T}$.
2. Update the transformation $\mathbf{T}$ by minimizing an objective function $E(\mathbf{T})$ defined over the correspondence set $\mathcal{K}$.

Different variants of ICP use different objective functions $E(\mathbf{T})$ [\[BeslAndMcKay1992\]](../reference.html#beslandmckay1992) [\[ChenAndMedioni1992\]](../reference.html#chenandmedioni1992) [\[Park2017\]](../reference.html#park2017).

We first show a point-to-point ICP algorithm [\[BeslAndMcKay1992\]](../reference.html#beslandmckay1992) using the objective

\begin{equation}
E(\mathbf{T}) = \sum_{(\mathbf{p},\mathbf{q})\in\mathcal{K}}\|\mathbf{p} - \mathbf{T}\mathbf{q}\|^{2}
\end{equation}

The class `TransformationEstimationPointToPoint` provides functions to compute the residuals and Jacobian matrices of the point-to-point ICP objective. The function `registration_icp` takes it as a parameter and runs point-to-point ICP to obtain the results.

In [8]:
source.points

std::vector<Eigen::Vector3d> with 113542 elements.
Use numpy.asarray() to access data.

In [10]:
print("Apply point-to-point ICP")
reg_p2p = o3d.pipelines.registration.registration_icp(
    source, target, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint())
print(reg_p2p)
print("Transformation is:")
print(reg_p2p.transformation)
# draw_registration_result(source, target, reg_p2p.transformation)

Apply point-to-point ICP
RegistrationResult with fitness=6.975392e-03, inlier_rmse=1.428228e-02, and correspondence_set size of 792
Access transformation to get result.
Transformation is:
[[ 9.99983939e-01  1.24666640e-03  5.52887078e-03 -3.07996446e+00]
 [-1.16817020e-03  9.99898804e-01 -1.41780864e-02  7.68892284e+00]
 [-5.54598662e-03  1.41714000e-02  9.99884200e-01 -9.85428192e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [11]:
import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

In [12]:
from utils.write_output import write_array2txt

In [15]:
# source_full = o3d.io.read_point_cloud("../../data/External Scans/tank_2/tank_2_005 cluster_5.txt", format='xyz')
source_aligned = copy.deepcopy(source).transform(reg_p2p.transformation)
write_array2txt(
    "D:\REGISTRATION DATA\External\JB-483 70ft EXT\\top_center_aligned_with_radius_extended_ground/057_aligned.txt",
    np.asarray(source_aligned.points)
)

array([[ 9.99993466e-01, -3.61395958e-03,  8.02159072e-05,
         1.11850038e-01],
       [ 3.61395972e-03,  9.99993470e-01, -1.62840349e-06,
        -5.79191936e-02],
       [-8.02094984e-05,  1.91828991e-06,  9.99999997e-01,
         6.77708822e-04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [43]:
source_aligned = copy.deepcopy(source).transform(reg_p2p.transformation)
write_array2txt(
    '../../data/External Scans/tank_2/tank_2_004 cluster_4 entire overlap - aligned.txt',
    np.asarray(source_aligned.points)
)

In [45]:
import pickle

In [58]:
with open('../../data/External Scans/trans56.pkl', 'wb') as f:
    pickle.dump(reg_p2p.transformation, f)

In [23]:
reg_p2p.transformation

array([[ 0.99996841, -0.00724514, -0.00327008,  0.11249931],
       [ 0.00724883,  0.9999731 ,  0.001119  ,  0.1753302 ],
       [ 0.00326188, -0.00114267,  0.99999403,  0.10302628],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

The `fitness` score increases from 0.174723 to 0.372450. The `inlier_rmse` reduces from 0.011771 to 0.007760. By default, `registration_icp` runs until convergence or reaches a maximum number of iterations (30 by default). It can be changed to allow more computation time and to improve the results further.

In [13]:
reg_p2p = o3d.pipelines.registration.registration_icp(
    source, target, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000))
print(reg_p2p)
print("Transformation is:")
print(reg_p2p.transformation)
# draw_registration_result(source, target, reg_p2p.transformation)

RegistrationResult with fitness=7.671170e-03, inlier_rmse=1.431430e-02, and correspondence_set size of 871
Access transformation to get result.
Transformation is:
[[ 9.99977937e-01  2.03659507e-04  6.63963679e-03 -3.58914950e+00]
 [-1.13216900e-04  9.99907249e-01 -1.36191509e-02  7.35188984e+00]
 [-6.64179463e-03  1.36180987e-02  9.99885210e-01 -9.18261678e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


The final alignment is tight. The `fitness` score improves to 0.621123. The `inlier_rmse` reduces to 0.006583.

In [8]:
import sys
sys.path.append('/Users/tinnguyen/BlackBox/PointRegistration/source')
from utils.write_output import write_array2txt

In [10]:
source_full = o3d.io.read_point_cloud("/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055_translated.txt", format='xyz')
source_aligned = copy.deepcopy(source_full).transform(reg_p2p.transformation)
write_array2txt(
    '/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055_aligned.txt',
    np.asarray(source_aligned.points)
)

## Point-to-plane ICP
The point-to-plane ICP algorithm [\[ChenAndMedioni1992\]](../reference.html#chenandmedioni1992) uses a different objective function

\begin{equation}
E(\mathbf{T}) = \sum_{(\mathbf{p},\mathbf{q})\in\mathcal{K}}\big((\mathbf{p} - \mathbf{T}\mathbf{q})\cdot\mathbf{n}_{\mathbf{p}}\big)^{2},
\end{equation}

where $\mathbf{n}_{\mathbf{p}}$ is the normal of point $\mathbf{p}$. [\[Rusinkiewicz2001\]](../reference.html#rusinkiewicz2001) has shown that the point-to-plane ICP algorithm has a faster convergence speed than the point-to-point ICP algorithm.

`registration_icp` is called with a different parameter `TransformationEstimationPointToPlane`. Internally, this class implements functions to compute the residuals and Jacobian matrices of the point-to-plane ICP objective.

In [3]:
print("Apply point-to-plane ICP")
reg_p2l = o3d.pipelines.registration.registration_icp(
    source, target, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPlane())
print(reg_p2l)
print("Transformation is:")
print(reg_p2l.transformation)
draw_registration_result(source, target, reg_p2l.transformation)

Apply point-to-plane ICP


RuntimeError: [Open3D Error] (class open3d::pipelines::registration::RegistrationResult __cdecl open3d::pipelines::registration::RegistrationICP(const class open3d::geometry::PointCloud &,const class open3d::geometry::PointCloud &,double,const class Eigen::Matrix<double,4,4,0,4,4> &,const class open3d::pipelines::registration::TransformationEstimation &,const class open3d::pipelines::registration::ICPConvergenceCriteria &)) D:\a\Open3D\Open3D\cpp\open3d\pipelines\registration\Registration.cpp:154: TransformationEstimationPointToPlane and TransformationEstimationColoredICP require pre-computed normal vectors for target PointCloud.


The point-to-plane ICP reaches tight alignment within 30 iterations (a `fitness` score of 0.620972 and an `inlier_rmse` score of 0.006581).

<div class="alert alert-info">
    
The point-to-plane ICP algorithm uses point normals. In this tutorial, we load normals from files. If normals are not given, they can be computed with [Vertex normal estimation](pointcloud.ipynb#vertex-normal-estimation).

</div>

In [1]:
from glob import glob
import os

In [6]:
from glob import glob
import os
data_dir = '/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output'
patch_id = 0
full_point = np.empty([0,4])
for fn in glob(os.path.join(data_dir, '*.txt')):
    if 'aligned' in fn:
        continue
    # read file
    ptc = np.zeros([10,3])
    # append id column to the right
    ptc = np.c_[ptc, patch_id * np.ones((ptc.shape[0], 1))]
    # append to the entire point cloud
    full_point = np.r_[full_point, ptc]
# save

/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_48.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_58.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_15.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_10.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_38.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_9.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_23.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_7.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_36.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_34.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/055-output/055 cluster_35.txt
/Users/tinnguyen/BlackBox/PointRegistration/data/JB-483/